In [5]:
# ===============================================
# 🎯 UK National Lottery – Full Analytical Report
# (PCA selected + PCA 3x3 grid with selected highlight)
# ===============================================

import fitz, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os, re
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from matplotlib.patches import Rectangle
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
from google.colab import drive

# === 1. Mount & Paths ===
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/datasets"
out_dir = os.path.join(folder, "lotto_kmeans_outputs")
os.makedirs(out_dir, exist_ok=True)
pdf_path = os.path.join(out_dir, "UK_Lotto_Executive_Summary_A4.pdf")
csv_path = os.path.join(out_dir, "lotto_full_validated.csv")
pdf_source = os.path.join(folder, "UK National Lotto Winning Numbers 2.pdf")

# === 2. Interactive Inputs ===
try:
    last_n_draws_input = input("Enter number of draws to analyze (or leave blank for full dataset): ")
    last_n_draws = int(last_n_draws_input) if last_n_draws_input.strip() else None
except:
    last_n_draws = None

try:
    k_input = input("Enter K for K-Means clustering (or leave blank for auto selection): ")
    user_defined_k = int(k_input) if k_input.strip() else None
except:
    user_defined_k = None

print(f"Analyzing last {last_n_draws or 'all'} draws. K-Means k = {user_defined_k or 'auto'}")

# === 3. PDF Extraction Function ===
def extract_lotto_table(pdf_path):
    doc = fitz.open(pdf_path)
    lines = []
    for page in doc:
        text = page.get_text("text")
        for line in text.split("\n"):
            if re.match(r"^\d{4,}", line.strip()):
                parts = [p.strip() for p in line.split(",") if p.strip()]
                if len(parts) >= 16:
                    day, dd, mmm, yyyy = parts[1], parts[2], parts[3], parts[4]
                    nums = parts[5:12]
                    jackpot, wins, machine, setnum = parts[12:16]
                    try:
                        date_str = f"{dd} {mmm} {yyyy}"
                        numbers = [int(n) for n in nums]
                        lines.append([date_str] + numbers + [int(jackpot), int(wins), machine, setnum])
                    except Exception:
                        # skip lines with unexpected formatting or conversion errors
                        continue
    cols = ["DrawDate","Ball1","Ball2","Ball3","Ball4","Ball5","Ball6","BN","Jackpot","Wins","Machine","Set"]
    return pd.DataFrame(lines, columns=cols)

# === 4. Load & Clean Data ===
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
else:
    df_existing = pd.DataFrame(columns=["DrawDate","Ball1","Ball2","Ball3","Ball4","Ball5","Ball6","BN","Jackpot","Wins","Machine","Set"])

df_new = extract_lotto_table(pdf_source)
# keep existing behaviour: concat then dedupe (if you prefer Option A replace csv entirely, change this)
df = pd.concat([df_existing, df_new]).drop_duplicates(subset=["DrawDate"], keep="last")
df['DrawDate'] = pd.to_datetime(df['DrawDate'], errors='coerce', dayfirst=True)
df = df.dropna(subset=['DrawDate']).sort_values("DrawDate").reset_index(drop=True)
df.to_csv(csv_path, index=False)
print(f"✅ Updated CSV with {len(df)} draws")

# === 5. Draw Selection (last N draws, ending at last actual draw) ===
last_draw_date = df['DrawDate'].max()
df_valid = df[df['DrawDate'] <= last_draw_date]

if last_n_draws is not None and last_n_draws > 0:
    df_range = df_valid.tail(last_n_draws).reset_index(drop=True)
    range_label = f"Last {last_n_draws} draws"
else:
    df_range = df_valid.copy()
    range_label = "Full dataset"

# === 6. Frequency & Reoccurrence Analysis ===
ball_cols = ["Ball1","Ball2","Ball3","Ball4","Ball5","Ball6"]
def safe_counts(series):
    s = pd.to_numeric(series, errors="coerce")
    s = s[(s >= 1) & (s <= 59)]
    return s.value_counts().sort_index()

global_main_counts = pd.Series(dtype=int)
for c in ball_cols:
    global_main_counts = global_main_counts.add(safe_counts(df_range[c]), fill_value=0)
global_main_counts = global_main_counts.fillna(0).astype(int).sort_index()

sorted_counts = global_main_counts.sort_values(ascending=False)
top_6_balls = sorted_counts.head(6)
bottom_6_balls = sorted_counts.tail(6)
middle_6_balls = sorted_counts.iloc[len(sorted_counts)//2 - 3 : len(sorted_counts)//2 + 3]

# Reoccurrence table data
reoccurrence_table_data = [["Category", "Ball", "Frequency"]]
for category, group in [("Most Frequent", top_6_balls), ("Middle", middle_6_balls), ("Least Frequent", bottom_6_balls)]:
    for ball, freq in group.items():
        reoccurrence_table_data.append([category, str(int(ball)), str(int(freq))])

# === 7. Moving Average Trend ===
trend_data = []
for num in range(1, 60):
    trend = [(df_range.iloc[:idx+1, 1:7] == num).sum().sum() for idx in range(len(df_range))]
    trend_data.append(trend)
trend_df = pd.DataFrame(trend_data, index=range(1,60), columns=df_range["DrawDate"])
trend_ma = trend_df.T.rolling(window=50, min_periods=1).mean()
trend_path = os.path.join(out_dir, "trendline.png")
plt.figure(figsize=(8,3))
plt.plot(trend_ma.index, trend_ma.mean(axis=1), color="red")
plt.title("50-draw Moving Average Frequency Trend")
plt.xlabel("Draw Number (Chronological)")
plt.ylabel("Average Frequency of Balls")
plt.tight_layout(); plt.savefig(trend_path, dpi=150); plt.close()

# === 8. K-Means & Clustering Metrics ===
num_data = df_range[ball_cols].dropna().astype(int)
sil_curve_path = None
elbow_path = None
heatmap_path = None
sil_path = None
cluster_labels = pd.Series(index=df_range.index, dtype=int)

# placeholders for PCA outputs
pca_selected_path = None
pca_grid_path = None

if len(num_data) < 2:
    print("⚠️ Not enough complete draws to perform clustering. Skipping K-Means.")
    silhouette = None
    k = None
    centroids = pd.DataFrame()
    silhouette_scores = []
else:
    max_k = min(10, len(num_data) - 1)
    silhouette_scores = []

    # Compute silhouette scores for K=2..max_k
    for k_test in range(2, max_k + 1):
        kmeans_tmp = KMeans(n_clusters=k_test, random_state=42, n_init=10)
        labels_tmp = kmeans_tmp.fit_predict(num_data)
        score = silhouette_score(num_data, labels_tmp)
        silhouette_scores.append((k_test, score))

    # Determine K
    if user_defined_k is None:
        k, silhouette = max(silhouette_scores, key=lambda x: x[1])
    else:
        k = user_defined_k
        labels_tmp = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(num_data)
        silhouette = silhouette_score(num_data, labels_tmp)

    # Final KMeans clustering
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(num_data)
    centroids = pd.DataFrame(kmeans.cluster_centers_, columns=ball_cols)
    cluster_labels[:] = labels

    # --- Silhouette Curve ---
    sil_curve_path = os.path.join(out_dir, "silhouette_curve.png")
    plt.figure(figsize=(6,3))
    ks = [x[0] for x in silhouette_scores]
    scores = [x[1] for x in silhouette_scores]
    plt.plot(ks, scores, marker='o', color='dodgerblue', label='Silhouette Score')
    if k in ks:
        idx = ks.index(k)
        plt.plot(ks[idx], scores[idx], marker='o', markersize=10, color='red', label=f'Selected K={k}')
        plt.annotate(f'{scores[idx]:.3f}', xy=(ks[idx], scores[idx]), xytext=(ks[idx]+0.2, scores[idx]-0.02),
                     color='red', fontsize=8)
    plt.title("Silhouette Score vs. Cluster Count (K)")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Silhouette Score")
    plt.xticks(ks)
    plt.legend()
    plt.tight_layout()
    plt.savefig(sil_curve_path, dpi=150)
    plt.close()

    # --- Elbow Curve ---
    elbow_path = os.path.join(out_dir, "elbow_curve.png")
    inertias = []
    for k_test in range(2, max_k + 1):
        kmeans_tmp = KMeans(n_clusters=k_test, random_state=42, n_init=10)
        kmeans_tmp.fit(num_data)
        inertias.append(kmeans_tmp.inertia_)

    plt.figure(figsize=(6,3))
    plt.plot(range(2, max_k+1), inertias, marker='o', color='dodgerblue', label='Inertia')
    if k in range(2, max_k+1):
        idx = k-2
        plt.plot(k, inertias[idx], marker='o', markersize=10, color='red', label=f'Selected K={k}')
        plt.annotate(f'{inertias[idx]:.0f}', xy=(k, inertias[idx]), xytext=(k+0.2, inertias[idx]*1.02),
                     color='red', fontsize=8)
    plt.title("Elbow Curve (Inertia vs. K)")
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Inertia (Sum of Squared Distances)")
    plt.xticks(range(2, max_k+1))
    plt.legend()
    plt.tight_layout()
    plt.savefig(elbow_path, dpi=150)
    plt.close()

    # --- Centroid Heatmap ---
    heatmap_path = os.path.join(out_dir, "centroid_heatmap.png")
    plt.figure(figsize=(8,4))
    sns.heatmap(centroids, annot=True, cmap="coolwarm", cbar=True)
    plt.title("K-Means Cluster Centroids (Average Ball Numbers per Position)")
    plt.xlabel("Ball Position")
    plt.ylabel("Cluster ID")
    plt.tight_layout()
    plt.savefig(heatmap_path, dpi=150)
    plt.close()

    # --- Cluster Avg Bar Chart ---
    sil_path = os.path.join(out_dir, "silhouette_bars.png")
    plt.figure(figsize=(6,3))
    plt.bar(range(1,k+1), [np.mean(num_data[labels==i].mean()) for i in range(k)], color="lightblue")
    plt.title(f"K-Means Cluster Averages (Silhouette={silhouette:.3f})")
    plt.xlabel("Cluster ID"); plt.ylabel("Avg Ball Value")
    plt.tight_layout(); plt.savefig(sil_path, dpi=150)
    plt.close()

    # === PCA computations ===
    try:
        pca = PCA(n_components=2, random_state=42)
        pca_data = pca.fit_transform(num_data)
    except Exception as e:
        pca_data = None
        print("PCA failed:", e)

    # --- PCA plot for selected K (single plot) ---
    if pca_data is not None:
        try:
            pca_selected_path = os.path.join(out_dir, f"pca_selected_k{k}.png")
            plt.figure(figsize=(6,5))
            plt.scatter(pca_data[:,0], pca_data[:,1], c=labels, s=12, alpha=0.8)
            plt.title(f"PCA Projection — Selected K={k}")
            plt.xlabel("PC1"); plt.ylabel("PC2")
            plt.tight_layout(); plt.savefig(pca_selected_path, dpi=200); plt.close()
        except Exception as e:
            print("Failed to save PCA selected plot:", e)
            pca_selected_path = None

        # --- PCA 3x3 grid (K=2..10) with highlight for selected K ---
        try:
            pca_grid_path = os.path.join(out_dir, "pca_kmeans_grid.png")
            fig, axes = plt.subplots(3,3, figsize=(12,12))
            axes = axes.flatten()
            for idx, k_plot in enumerate(range(2, 11)):
                ax = axes[idx]
                try:
                    labels_k = KMeans(n_clusters=k_plot, random_state=42, n_init=10).fit_predict(num_data)
                    ax.scatter(pca_data[:,0], pca_data[:,1], c=labels_k, s=8, alpha=0.7)
                except Exception:
                    ax.scatter(pca_data[:,0], pca_data[:,1], s=8, alpha=0.7, color='grey')
                ax.set_title(f"K = {k_plot}")
                ax.set_xticks([]); ax.set_yticks([])

                if k_plot == k:
                    rect = Rectangle((0,0),1,1, transform=ax.transAxes, linewidth=3, edgecolor="red", facecolor="none")
                    ax.add_patch(rect)
                    ax.text(0.5, -0.12, "SELECTED", transform=ax.transAxes, ha="center", color="red", fontsize=10, fontweight="bold")

            fig.suptitle("PCA Projection of Lottery Draws (K = 2–10)", fontsize=14)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.savefig(pca_grid_path, dpi=200)
            plt.close(fig)
        except Exception as e:
            print("Failed to create PCA grid:", e)
            pca_grid_path = None

# === 9. Frequency Histogram ===
hist_main_path = os.path.join(out_dir, "hist_main.png")
plt.figure(figsize=(8,3))
plt.bar(global_main_counts.index, global_main_counts.values, color="dodgerblue", edgecolor="black")
plt.title(f"Ball Frequency – {range_label}")
plt.xlabel("Ball Number"); plt.ylabel("Frequency")
plt.tight_layout(); plt.savefig(hist_main_path, dpi=150); plt.close()
print("✅ Charts generated and saved")

# === 10. PDF Generation (UK date format, all charts + PCA visualisations + methodology) ===
styles = getSampleStyleSheet()
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
story = []

# --- Executive Summary ---
story.append(Paragraph("<b>Executive Summary</b>", styles['Title']))
story.append(Spacer(1, 0.25*inch))
story.append(Paragraph(f"Total draws analysed: {len(df_range)}", styles['Heading2']))
story.append(Paragraph(f"Date range: {df_range['DrawDate'].min().strftime('%d/%m/%Y')} to {df_range['DrawDate'].max().strftime('%d/%m/%Y')}", styles['BodyText']))
story.append(Spacer(1, 0.15*inch))

# --- Methodology ---
method_text = """
<b>Methodology — Clustering & Model Selection</b><br/>
K-Means clustering is applied to the six main drawn numbers for each draw to explore potential groupings.
The primary quantitative selection metric is the mean silhouette score evaluated for K = 2..10.
Because lottery draws are high-noise and near-random, absolute silhouette values are expected to be low; we therefore
use per-K diagnostic visualisations (per-sample silhouette distributions, cluster size balance, centroid dispersion)
and PCA projections for interpretability. The selected K is chosen to maximise silhouette while also ensuring interpretable and reasonably balanced clusters.
PCA is solely used for visualization and does not affect model fitting.
"""
story.append(Paragraph(method_text, styles['BodyText']))
story.append(Spacer(1, 0.2*inch))

# --- Ball Frequency Overview ---
story.append(Paragraph("<b>Ball Frequency Overview</b>", styles['Heading3']))
story.append(Paragraph(f"Most Frequent Balls: {', '.join(map(str, top_6_balls.index))}", styles['BodyText']))
story.append(Paragraph(f"Middle Frequent Balls: {', '.join(map(str, middle_6_balls.index))}", styles['BodyText']))
story.append(Paragraph(f"Least Frequent Balls: {', '.join(map(str, bottom_6_balls.index))}", styles['BodyText']))
story.append(Spacer(1, 0.2*inch))

# --- Section 1: Recent 20 Draws with Cluster ---
preview = df_range.tail(20).copy().iloc[::-1]
preview['DrawDate'] = preview['DrawDate'].dt.strftime('%d/%m/%Y')
preview['Cluster'] = cluster_labels.tail(20).iloc[::-1].astype(str).values
table_data = [preview.columns.tolist()] + preview.astype(str).values.tolist()
table = Table(table_data, repeatRows=1)
table.setStyle(TableStyle([
    ('BACKGROUND',(0,0),(-1,0),colors.lightgrey),
    ('ALIGN',(0,0),(-1,-1),'CENTER'),
    ('FONTSIZE',(0,0),(-1,-1),7),
    ('INNERGRID',(0,0),(-1,-1),0.25,colors.grey),
    ('BOX',(0,0),(-1,-1),0.25,colors.black),
]))
story.append(Paragraph("<b>1. Recent 20 Draws (Newest First, with Cluster)</b>", styles['Heading2']))
story.append(table)
story.append(Spacer(1, 0.25*inch))

# --- Section 2: Ball Reoccurrence Table ---
table_re = Table(reoccurrence_table_data, repeatRows=1)
table_re.setStyle(TableStyle([
    ('BACKGROUND',(0,0),(-1,0),colors.lightgrey),
    ('ALIGN',(0,0),(-1,-1),'CENTER'),
    ('FONTSIZE',(0,0),(-1,-1),8),
    ('INNERGRID',(0,0),(-1,-1),0.25,colors.grey),
    ('BOX',(0,0),(-1,-1),0.25,colors.black),
]))
story.append(Paragraph("<b>2. Ball Reoccurrence Analysis</b>", styles['Heading2']))
story.append(table_re)
story.append(Spacer(1, 0.25*inch))

# --- Section 3: K-Means Charts ---
story.append(Paragraph("<b>3. K-Means Clustering Analysis</b>", styles['Heading2']))
if k is None or silhouette is None:
    story.append(Paragraph("⚠️ Not enough valid data to perform clustering.", styles['BodyText']))
else:
    story.append(Paragraph(f"K-Means performed with k={k} clusters. Silhouette Score = {silhouette:.3f}.", styles['BodyText']))
    for img_path in [sil_curve_path, elbow_path, heatmap_path, sil_path]:
        if img_path and os.path.exists(img_path):
            story.append(Image(img_path, width=6.3*inch, height=2.5*inch))
story.append(Spacer(1, 0.25*inch))

# --- Insert PCA selected plot and PCA grid ---
if pca_selected_path and os.path.exists(pca_selected_path):
    story.append(Paragraph("<b>PCA Projection — Selected K</b>", styles['Heading3']))
    story.append(Image(pca_selected_path, width=6.3*inch, height=4.8*inch))
    story.append(Spacer(1, 0.15*inch))

if pca_grid_path and os.path.exists(pca_grid_path):
    story.append(Paragraph("<b>PCA Grid — Comparison Across K (2–10)</b>", styles['Heading3']))
    story.append(Paragraph("Selected K is highlighted in red on the grid.", styles['BodyText']))
    story.append(Image(pca_grid_path, width=6.3*inch, height=6.3*inch))
    story.append(Spacer(1, 0.25*inch))

# --- Section 4: Frequency Distributions ---
story.append(Paragraph("<b>4. Frequency Distribution</b>", styles['Heading2']))
for img_path in [hist_main_path, trend_path]:
    if img_path and os.path.exists(img_path):
        story.append(Image(img_path, width=6.3*inch, height=2.5*inch))
story.append(Spacer(1, 0.25*inch))

# --- Section 5: Conclusions ---
story.append(Paragraph("<b>5. Conclusions</b>", styles['Heading2']))
if silhouette:
    story.append(Paragraph(f"• Optimal cluster count: {k} (Silhouette={silhouette:.3f}).", styles['BodyText']))
story.append(Paragraph("• Ball frequency distribution shows mild decade bias.", styles['BodyText']))
story.append(Paragraph("• No repeating 6-number lines observed in full dataset.", styles['BodyText']))

doc.build(story)
print(f"✅ Full analytical PDF generated → {pdf_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter number of draws to analyze (or leave blank for full dataset): 
Enter K for K-Means clustering (or leave blank for auto selection): 
Analyzing last all draws. K-Means k = auto


/tmp/ipython-input-1188678474.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df_existing, df_new]).drop_duplicates(subset=["DrawDate"], keep="last")
/tmp/ipython-input-1188678474.py:74: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['DrawDate'] = pd.to_datetime(df['DrawDate'], errors='coerce', dayfirst=True)


✅ Updated CSV with 3116 draws
✅ Charts generated and saved
✅ Full analytical PDF generated → /content/drive/MyDrive/datasets/lotto_kmeans_outputs/UK_Lotto_Executive_Summary_A4.pdf
